<a href="https://colab.research.google.com/github/tomasndlate/ai-learning/blob/main/prod_pipeline_vlm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install dependencies

In [ ]:
# Install/Update Core Hugging Face libraries
!pip install -q --upgrade transformers accelerate

# Hard Requirements for SmolVLM2 Tokenizers & Video Frame handling
!pip install -q num2words decord av

# Memory Optimization (Crucial for loading-in-8bit or 4bit on standard Colab GPUs)
!pip install -q bitsandbytes

print('Done. Code dependencies installed successfully.')

Pipeline

In [ ]:
import torch
import time
from PIL import Image
from PIL.Image import Resampling
from transformers import AutoProcessor, AutoModelForImageTextToText

class SmolVLM2InferencePipeline:
    def __init__(self, model_path="HuggingFaceTB/SmolVLM2-2.2B-Instruct"):
        print("Initializing production pipeline... Loading model into memory.")
        self.PATCH_SIZE = 384
        self.processor = AutoProcessor.from_pretrained(model_path)
        self.processor.image_processor.max_pixels = self.PATCH_SIZE * self.PATCH_SIZE

        # Load model with performance-optimized configurations from your research
        self.model = AutoModelForImageTextToText.from_pretrained(
            model_path,
            torch_dtype=torch.bfloat16,
            _attn_implementation="sdpa", # Fast attention optimization
            device_map="auto"
        )
        print("Pipeline successfully loaded and ready.")

    def preprocess_image(self, image_path):
        """Applies your custom aspect-ratio locking research math to the raw input."""
        loaded_image = Image.open(image_path).convert("RGB")
        orig_width, orig_height = loaded_image.size

        # Your custom thesis math: Lock height to 1 patch size, dynamically scale width
        scale_factor = self.PATCH_SIZE / orig_height
        new_width = int(orig_width * scale_factor)
        new_height = self.PATCH_SIZE

        return loaded_image.resize((new_width, new_height), Resampling.BILINEAR)

    def run_inference(self, image_path, prompt):
        """Processes a single payload from start to finish."""
        optimized_image = self.preprocess_image(image_path)

        # Standardize conversational layout
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": prompt}
                ]
            }
        ]

        # Tokenize inputs onto target GPU execution space
        inputs = self.processor(
            images=optimized_image,
            text=self.processor.apply_chat_template(messages, add_generation_prompt=True),
            return_tensors="pt",
            do_resize=False,
            do_pad=False
        ).to(self.model.device)

        # Generate output deterministically
        with torch.no_grad():
            generated_ids = self.model.generate(
                **inputs,
                max_new_tokens=80,
                use_cache=True,     # Token caching reduces latency significantly
                do_sample=False     # Deterministic execution
            )

        # Decode and isolate target output string
        generated_text = self.processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        response_body = generated_text.split("assistant\n")[-1].strip()
        return response_body

# Test run to ensure your production pipeline class is working flawlessly
pipeline = SmolVLM2InferencePipeline()

Build and Deploy

In [ ]:
# Save model and processor out to a clean localized directory inside your drive
export_directory = "/content/drive/MyDrive/thesis/exported_smolvlm2"

pipeline.model.save_pretrained(export_directory)
pipeline.processor.save_pretrained(export_directory)
print(f"Hugging Face weights and token structures compiled successfully to: {export_directory}")

In [ ]:
# 1. Compile the native llama.cpp quantization command-line engine
!cd llama.cpp && make llama-quantize -j

# 2. Convert the Hugging Face weights to an uncompressed F16 GGUF Language Model
!python3 llama.cpp/convert_hf_to_gguf.py /content/drive/MyDrive/thesis/exported_smolvlm2 \
  --outfile /content/drive/MyDrive/thesis/smolvlm2-text-f16.gguf \
  --outtype f16

# 3. Use the compiled engine to compress that F16 file down to 4-bit Q4_K_M
!./llama.cpp/llama-quantize /content/drive/MyDrive/thesis/smolvlm2-text-f16.gguf /content/drive/MyDrive/thesis/smolvlm2-text.gguf Q4_K_M

# 4. Convert the Hugging Face weights to the F16 Vision Projector Component
!python3 llama.cpp/convert_hf_to_gguf.py /content/drive/MyDrive/thesis/exported_smolvlm2 \
  --mmproj \
  --outfile /content/drive/MyDrive/thesis/smolvlm2-vision.gguf \
  --outtype f16